In [6]:
import collections
import os
import json
import math
import pandas as pd

# --- Configuración de rutas de directorios ---
results_dir = "data_for_seeds"

# Escanear y filtrar archivos solo para n=33
json_files = [
    f for f in os.listdir(results_dir) 
    if f.startswith("simulation_data_") and "_n33_" in f and f.endswith(".json")
]

rows_accumulator = []

for current_file in sorted(json_files):
    file_path = os.path.join(results_dir, current_file)
    
    with open(file_path, "r") as f:
        payload = json.load(f)
    
    meta = payload["meta_parameters"]
    beta_val = meta["beta"]
    m_val = meta["m"]
    n_val = meta["n"]
    
    for instance in payload["results"]:
        instance_id = instance["instance_id"]
        
        # Recuperar estructuras nativas del JSON
        sorted_cols = instance["sorted_cols"]
        posteriors = instance["posteriors"]
        posterior_matrix = instance["posterior_matrix"]  # Tamaño: m x n
        E_tilde_matrix = instance["E_tilde_matrix"]      # Tamaño: m x n (Ruidosa)
        E_original = instance["E_original"]              # Tamaño: m x n (Original/Real)
        
        # =====================================================================
        # TAREA 1: Procesamiento por Columnas Completas (Top 8 MAP)
        # =====================================================================
        top_8_cols = sorted_cols[:8]
        
        col_bit_strings = []
        real_col_bit_strings = []
        global_col_score = 1.0
        
        for col_idx in top_8_cols:
            col_str_key = str(col_idx)
            score, col_raw_str = posteriors[col_str_key]
            
            # String RUIDOSO (E_tilde_matrix)
            clean_bits = col_raw_str.replace("[", "").replace("]", "").replace(",", "").replace(" ", "")
            col_bit_strings.append(clean_bits)
            
            # String REAL (E_original) reconstruido para la misma columna
            real_bits = "".join(str(E_original[i][col_idx]) for i in range(m_val))
            real_col_bit_strings.append(real_bits)
            
            # Multiplicar verosimilitudes para el score global
            global_col_score *= float(score)
            
        # Concatenación de strings observados y reales
        concatenated_cols_str = "".join(col_bit_strings)
        concatenated_real_cols_str = "".join(real_col_bit_strings)
        
        # =====================================================================
        # TAREA 2: Procesamiento por Entradas de Bits Individuales (Top 256)
        # =====================================================================
        all_entries = []
        for i in range(m_val):        # Filas (0 a m-1)
            for j in range(n_val):    # Columnas (0 a n-1)
                bit_score = posterior_matrix[i][j]
                observed_bit = int(E_tilde_matrix[i][j])
                all_entries.append(((i, j), bit_score, observed_bit))
                
        all_entries_sorted = sorted(all_entries, key=lambda x: x[1], reverse=True)
        top_256_entries = all_entries_sorted[:256]
        
        entry_indices = []
        entry_bits = []
        real_entry_bits = []
        global_entry_score = 1.0
        
        for (i, j), score, obs_bit in top_256_entries:
            entry_indices.append((i, j))
            
            # Bit RUIDOSO
            entry_bits.append(str(obs_bit))
            
            # Bit REAL correspondiente a la misma posición (i, j) en E_original
            real_bit = str(E_original[i][j])
            real_entry_bits.append(real_bit)
            
            global_entry_score *= float(score)
            
        concatenated_entries_str = "".join(entry_bits)
        concatenated_real_entries_str = "".join(real_entry_bits)
        
        # =====================================================================
        # Consolidar datos en el registro para el DataFrame
        # =====================================================================
        record = {
            "beta": beta_val,
            "instance_id": instance_id,
            "n": n_val,
            "k": meta["k"],
            "r": meta["r"],
            "m": m_val,
            
            # Columnas completas
            "col_top_8_indices": top_8_cols,
            "col_concatenated_string": concatenated_cols_str,
            "col_real_concatenated_string": concatenated_real_cols_str,
            "col_global_score": global_col_score,
            
            # Bits individuales
            "entry_top_256_indices": entry_indices,
            "entry_concatenated_string": concatenated_entries_str,
            "entry_real_concatenated_string": concatenated_real_entries_str,
            "entry_global_score": global_entry_score
        }
        rows_accumulator.append(record)

# Crear el DataFrame de Pandas final
df_analysis = pd.DataFrame(rows_accumulator)

# --- Verificación en Consola ---
print(f"[SUCCESS] DataFrame generado. Total filas: {len(df_analysis)}")
print("\nPrimeras filas del análisis consolidado con comparativa Ruidosa vs Real:")
print(df_analysis[[
    "beta", 
    "instance_id", 
    "col_global_score", 
    "entry_global_score", 
    "col_concatenated_string", 
    "col_real_concatenated_string", 
    "entry_concatenated_string", 
    "entry_real_concatenated_string"
]].head())

[SUCCESS] DataFrame generado. Total filas: 250

Primeras filas del análisis consolidado con comparativa Ruidosa vs Real:
   beta  instance_id  col_global_score  entry_global_score  \
0  0.03            1          0.013691            0.035055   
1  0.03            2          0.013550            0.035055   
2  0.03            3          0.014123            0.035055   
3  0.03            4          0.014872            0.035055   
4  0.03            5          0.013691            0.035055   

                             col_concatenated_string  \
0  0010010000011010100101110100001011000010001011...   
1  1010010101010001000000011000001000001001110010...   
2  1100100100110001000000010001001100100110010010...   
3  0010100000000100010101001011000001100010110000...   
4  0011011010100101000001000000011100100100000000...   

                        col_real_concatenated_string  \
0  0010010000011010100101110100001011000011001011...   
1  1010010100010001000000001000001000001000110010...   
2

In [7]:
df_analysis[["beta", "instance_id", "col_global_score", "entry_global_score", "col_concatenated_string", "entry_concatenated_string", "col_top_8_indices", "entry_top_256_indices"]].head()

,beta,instance_id,col_global_score,entry_global_score,col_concatenated_string,entry_concatenated_string,col_top_8_indices,entry_top_256_indices
0,0.03,1,0.013691,0.035055,0010010000011010100101110100001011000010001011...,0000000000000000000000000000000000000000000000...,"[17, 7, 32, 2, 1, 9, 22, 23]","[(0, 2), (0, 4), (0, 7), (0, 9), (0, 10), (0, ..."
1,0.03,2,0.013550,0.035055,1010010101010001000000011000001000001001110010...,0000000000000000000000000000000000000000000000...,"[17, 12, 31, 10, 15, 2, 20, 4]","[(0, 0), (0, 2), (0, 3), (0, 4), (0, 6), (0, 9..."
2,0.03,3,0.014123,0.035055,1100100100110001000000010001001100100110010010...,0000000000000000000000000000000000000000000000...,"[18, 29, 8, 32, 0, 13, 19, 4]","[(0, 0), (0, 2), (0, 3), (0, 4), (0, 5), (0, 7..."
3,0.03,4,0.014872,0.035055,0010100000000100010101001011000001100010110000...,0000000000000000000000000000000000000000000000...,"[2, 6, 4, 21, 1, 24, 9, 28]","[(0, 2), (0, 3), (0, 4), (0, 6), (0, 16), (0, ..."
4,0.03,5,0.013691,0.035055,0011011010100101000001000000011100100100000000...,0000000000000000000000000000000000000000000000...,"[4, 18, 8, 9, 27, 29, 31, 22]","[(0, 1), (0, 4), (0, 6), (0, 9), (0, 11), (0, ..."


In [8]:
first = df_analysis[["beta", "instance_id", "col_global_score", "entry_global_score", "col_concatenated_string", "entry_concatenated_string", "col_top_8_indices", "entry_top_256_indices"]].iloc[4]
first

beta                                                                      0.03
instance_id                                                                  5
col_global_score                                                      0.013691
entry_global_score                                                    0.035055
col_concatenated_string      0011011010100101000001000000011100100100000000...
entry_concatenated_string    0000000000000000000000000000000000000000000000...
col_top_8_indices                                [4, 18, 8, 9, 27, 29, 31, 22]
entry_top_256_indices        [(0, 1), (0, 4), (0, 6), (0, 9), (0, 11), (0, ...
Name: 4, dtype: object

In [ ]:
from seed_recovery_algorithms import *
from tqdm import tqdm

def recovery_pipeline_row(row, target_col, alpha, w, eta, mu, Btime=2**30, Bmemory=2**30, Cbase=2**5, Cblock=2**4, Coracle=2**6):
    """
    Executes the seed recovery pipeline on a single row configuration.
    
    Input:
        - row: A pandas Series representing a single dataframe entry
        - target_col: str defining which string column to process ('col_concatenated_string' or 'entry_concatenated_string')
        - alpha: float representing the probability of 0 flipping to 1
        - w: int defining the bit width of an individual chunk
        - eta: int defining the chunk group aggregation dimension
        - mu: int defining the maximum number of top candidates retained per block group
        
    Output:
        - str representing the recovered seed candidate, or None if extraction fails
    """
    bits = row[target_col]
    
    # Handle missing or malformed inputs gracefully
    if not isinstance(bits, str) or len(bits) == 0:
        return None
        
    bits_list = [int(char) for char in bits]
    beta = row["beta"]
    W = len(bits)
    
    # 1. Pipeline preparation and execution
    P = build_posteriors_from_tilde(bits_list, alpha, beta)
    L = generate_candidates(P, W, w, eta, mu, scale=10000.0)
    
    # 2. Score band resolution
    B_min = sum(min(cand.score for cand in block) for block in L)
    B_max = sum(max(cand.score for cand in block) for block in L)

    B1 = int(B_min)
    B2 = findOptimalB2(L, B1, int(B_max), W, w, eta, mu, Btime, Bmemory, Cbase, Cblock, Coracle, scale=10000)
    
    # 3. Dynamic programming lattice population
    B = create(L, int(B1), int(B2), W, w, eta, mu, scale=10000)
    B_0 = rank(L, B1, B2, W, w, eta, mu)
    
    # 4. Sequential path traversal (FIXED: B_0 + 1 to include the terminal rank index)
    for r in range(1, B_0 + 1):
        candidate_seed = getSeed(L, B, B1, B2, W, w, eta, mu, r=r)
        if candidate_seed is not None:
            if len(candidate_seed) == W: 
                return candidate_seed   
                
    return None

In [10]:
# Constants configurations
alpha = 0.01
w = 4
eta = 2
mu = 64

print("Processing 'col_concatenated_string' columns...")
df_analysis["recovered_seed_col"] = df_analysis.apply(
    lambda row: recovery_pipeline_row(row, "col_concatenated_string", alpha, w, eta, mu), 
    axis=1
)

print("Processing 'entry_concatenated_string' columns...")
df_analysis["recovered_seed_entry"] = df_analysis.apply(
    lambda row: recovery_pipeline_row(row, "entry_concatenated_string", alpha, w, eta, mu), 
    axis=1
)

Processing 'col_concatenated_string' columns...


TypeError: 'float' object cannot be interpreted as an integer

In [ ]:
df_analysis.to_csv('seed_results.csv')